# TP 11 — Vision par ordinateur distribuée### Module 6 — Stocker des images, distribuer un modèle> **Filière Art Numérique.** La filière Ingénierie Financière traite le **TP10**> (pipeline de détection de fraude) pendant que vous faites celui-ci.>> Le corrigé du TP10 vous sera distribué : **ses concepts sont au programme de> l'examen**, sa mise en œuvre ne l'est pas. Deux points à en retenir en> particulier — la fuite de données, et le choix d'un seuil de décision à partir> d'un coût métier.**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin1. **Mesurer** l'écart entre fichiers individuels et images empaquetées en Parquet.2. **Mesurer** trois façons de charger un modèle dans une UDF, et expliquer l'écart.3. Écrire une `pandas_udf` à itérateur, et dire pourquoi cette forme est la bonne.4. Appliquer une **classification** puis une **détection**, et écrire le résultat en Iceberg.5. Dimensionner une inférence par le calcul, et vérifier que le stockage suit.6. Observer un **décalage de distribution** et en tirer les conséquences.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Le jeu d'images et le modèle | 2 || 2 | Fichiers contre Parquet | 4 || 3 | Trois façons de charger le modèle | 6 || 4 | Classification et écriture en Iceberg | 3 || 5 | Détection d'objets | 3 || 6 | Dimensionner, et le décalage de distribution | 2 |## Section 0 — Deux modes d'exécutionCe TP fonctionne dans deux modes, et **les exercices 2, 3 et 6 sont identiques dans lesdeux** — ce sont ceux qui portent sur la distribution.| Mode | Condition | Ce qui change ||---|---|---|| **Réel** | Les modèles ONNX sont présents | Les prédictions ont un sens (relatif — voir Q6) || **Substitution** | Aucun modèle disponible | Un modèle factice simule le coût d'une inférence |Pour passer en mode réel, une seule fois et sur une connexion confortable :```bashpython 99-Infra/scripts/telecharger_modeles.py --sortie 99-Infra/modeles```

---# Exercice 1 — Le jeu d'images et le modèle  *(2 points)*

In [ ]:
# 1.1 — Session Sparkfrom pyspark.sql import SparkSession, functions as Ffrom pyspark.sql.types import (StructType, StructField, StringType, FloatType,                               ArrayType, IntegerType)from pyspark.sql.functions import pandas_udf, udffrom typing import Iteratorimport pandas as pd, numpy as np, io, os, time, subprocess, globUTILISATEUR = "etudiant"spark = (SparkSession.builder    .appName("TP11 - vision distribuee")    .master("local[4]")    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")    .config("spark.sql.execution.arrow.pyspark.enabled", "true")    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "256")    .config("spark.sql.shuffle.partitions", "8")    .getOrCreate())spark.sparkContext.setLogLevel("WARN")print("Spark", spark.version, "| UI :", spark.sparkContext.uiWebUrl)

In [ ]:
# 1.2 — Générer le jeu d'images (2 à 3 minutes)DOSSIER = "/home/tinku/work/images"!python /home/tinku/cours/99-Infra/scripts/generate_images.py \        --sortie {DOSSIER} --nombre 4000 --taille 224!ls {DOSSIER} | head -3!echo "..." && ls {DOSSIER} | wc -l && echo "images"!du -sh {DOSSIER}

In [ ]:
# 1.3 — Quel mode ? On regarde ce qui est disponible.MODELES = "/home/tinku/cours/99-Infra/modeles"CLASSIF  = f"{MODELES}/mobilenetv2-12.onnx"DETECT   = f"{MODELES}/ssd_mobilenet_v1_10.onnx"try:    import onnxruntime as ort    ONNX_DISPONIBLE = Trueexcept ImportError:    ONNX_DISPONIBLE = FalseMODE_REEL = ONNX_DISPONIBLE and os.path.exists(CLASSIF)print("onnxruntime installe :", ONNX_DISPONIBLE)print("modele classification :", os.path.exists(CLASSIF))print("modele detection      :", os.path.exists(DETECT))print("\n>>> MODE :", "REEL" if MODE_REEL else "SUBSTITUTION")if not MODE_REEL:    print("    Les exercices 2, 3 et 6 restent entierement valables.")

In [ ]:
# 1.4 — Le "modèle" de substitution : il simule le coût, pas l'intelligenceCOUT_CHARGEMENT_S = 0.6      # ce que coute le chargement d'un vrai modeleCOUT_IMAGE_MS     = 25       # ce que coute une inference par imageclass ModeleSubstitution:    """Simule un modele : chargement lent, inference proportionnelle au lot."""    def __init__(self, chemin=None):        time.sleep(COUT_CHARGEMENT_S)          # le chargement coute cher        self.classes = ["cercle", "carre", "triangle", "etoile", "anneau"]    def predire(self, images_octets):        n = len(images_octets)        time.sleep(n * COUT_IMAGE_MS / 1000.0)  # inference proportionnelle        # prediction deterministe a partir des octets : reproductible        return [self.classes[(len(b) if b else 0) % 5] for b in images_octets]def charger_modele(chemin=CLASSIF):    """Retourne un objet dote d'une methode .predire(list[bytes]) -> list[str]."""    if MODE_REEL:        return ModeleONNX(chemin)    return ModeleSubstitution(chemin)print("modele de substitution pret")

### Q1 *(2 pts)* —- **a.** Combien d'images, pour quel volume total ? Quelle taille moyenne par fichier ?- **b.** Comparez cette taille moyenne à la recommandation du module 2 sur la taille des  fichiers. De quel facteur êtes-vous en dessous ?- **c.** Sur un stockage objet facturant 0,0004 € les mille requêtes GET, combien coûterait  **un** parcours complet de ce jeu ? Et d'un jeu de 8 millions d'images ?

**Votre réponse :***(rédigez ici)*

---# Exercice 2 — Fichiers contre Parquet  *(4 points)***Objectif.** Mesurer ce que coûte le stockage en fichiers individuels, et ce que rapportel'empaquetage.

## 2.1 — **PRÉDICTION** *(1 pt)*On va lire les 4 000 images de deux façons : depuis les fichiers individuels, puis depuis unParquet qui les contient toutes.Laquelle sera la plus rapide ? De quel ordre de grandeur ? Et surtout : **quelle métrique dela Spark UI** montrera la différence ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 2.2 — Lecture en fichiers individuelsdef chrono(libelle, fonction):    debut = time.time(); r = fonction(); d = time.time() - debut    print(f"{libelle:<42} {d:7.2f} s"); return d, rspark.sparkContext.setJobDescription("A - binaryFile")images_fichiers = (spark.read.format("binaryFile")                   .option("pathGlobFilter", "*.jpg")                   .load(f"file://{DOSSIER}"))print("partitions :", images_fichiers.rdd.getNumPartitions())t_fichiers, n1 = chrono("lecture depuis fichiers individuels",                        lambda: images_fichiers.count())images_fichiers.select("path", "length").show(3, truncate=False)

In [ ]:
# 2.3 — Empaqueter en ParquetCHEMIN_PQ = f"hdfs://namenode:8020/user/{UTILISATEUR}/images_parquet"spark.sparkContext.setJobDescription("B - empaquetage")t_pack, _ = chrono("empaquetage en Parquet", lambda:    (images_fichiers     .withColumn("nom", F.element_at(F.split("path", "/"), -1))     .select("nom", "length", F.col("content").alias("image"))     .repartition(4)                    # peu de fichiers, gros     .write.mode("overwrite").parquet(CHEMIN_PQ)))def taille_hdfs(c):    o = subprocess.run(["hdfs","dfs","-du","-s",c], capture_output=True, text=True).stdout.split()    return int(o[0])/1024**2 if o else 0.0!hdfs dfs -ls {CHEMIN_PQ} | grep -c parquetprint("taille Parquet :", round(taille_hdfs(CHEMIN_PQ), 1), "Mio")

In [ ]:
# 2.4 — Relire depuis Parquetspark.sparkContext.setJobDescription("C - lecture Parquet")images_pq = spark.read.parquet(CHEMIN_PQ)print("partitions :", images_pq.rdd.getNumPartitions())t_parquet, n2 = chrono("lecture depuis Parquet", lambda: images_pq.count())print(f"\n{n1} images depuis les fichiers, {n2} depuis Parquet")print(f"rapport de duree : {t_fichiers/t_parquet:.2f}x")

### Q2 *(3 pts)* — Ouvrez la Spark UI, onglet **Stages**, et comparez les jobs A et C.- **a.** Combien de tâches pour chacun ? Quel rapport ?- **b.** Quelle est la durée médiane d'une tâche dans chaque cas ? Que reconnaissez-vous ?- **c.** L'écart de durée est modeste ici. Pourquoi ? Que se passerait-il sur un stockage  objet distant, et sur 8 millions d'images ?

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — Trois façons de charger le modèle  *(6 points)***L'exercice central du TP.** Les trois cellules calculent la même chose. Elles diffèrentuniquement par **le moment où le modèle est chargé**.

## 3.1 — **PRÉDICTION** *(1 pt)*Le chargement du modèle coûte **0,6 seconde**. On traite 4 000 images, en lots de 256, sur4 cœurs.Pour chacune des trois formes ci-dessous, dites **combien de fois le modèle sera chargé** etcombien de secondes cela représentera :1. UDF simple — `charger_modele()` appelé dans le corps de la fonction2. `pandas_udf` **sans** itérateur — `pd.Series -> pd.Series`3. `pandas_udf` **à** itérateur — `Iterator[pd.Series] -> Iterator[pd.Series]`

**Votre réponse :***(rédigez ici)*

In [ ]:
# 3.2 — FORME 1 : UDF simple (sur un échantillon seulement !)# ATTENTION : sur les 4 000 images cette forme prendrait ~40 minutes.# On la mesure donc sur 100 images, puis on extrapole.echantillon = images_pq.limit(100).cache()echantillon.count()@udf(returnType=StringType())def classer_naif(image):    m = charger_modele()                 # <-- UNE FOIS PAR LIGNE    return m.predire([image])[0]spark.sparkContext.setJobDescription("D - UDF simple (100 images)")t_naif, _ = chrono("forme 1 : UDF simple, 100 images",                   lambda: echantillon.withColumn("c", classer_naif("image")).count())print(f"  extrapolation a 4 000 images : {t_naif * 40:.0f} s "      f"({t_naif * 40 / 60:.1f} min)")

In [ ]:
# 3.3 — FORME 2 : pandas_udf SANS itérateur@pandas_udf(StringType())def classer_lot(images: pd.Series) -> pd.Series:    m = charger_modele()                 # <-- UNE FOIS PAR LOT    return pd.Series(m.predire(list(images)))spark.sparkContext.setJobDescription("E - pandas_udf sans iterateur")t_lot, _ = chrono("forme 2 : pandas_udf par lot, 4 000 images",                  lambda: images_pq.withColumn("c", classer_lot("image")).count())

In [ ]:
# 3.4 — FORME 3 : pandas_udf À ITÉRATEUR — la bonne@pandas_udf(StringType())def classer_iter(lots: Iterator[pd.Series]) -> Iterator[pd.Series]:    m = charger_modele()                 # <-- UNE FOIS PAR PROCESSUS    for lot in lots:                     # <-- puis on boucle sur les lots        yield pd.Series(m.predire(list(lot)))spark.sparkContext.setJobDescription("F - pandas_udf a iterateur")t_iter, _ = chrono("forme 3 : pandas_udf a iterateur, 4 000 images",                   lambda: images_pq.withColumn("c", classer_iter("image")).count())print(f"\n{'forme':<34}{'duree':>10}{'rapport':>10}")print("-" * 54)print(f"{'1. UDF simple (extrapole)':<34}{t_naif*40:>9.1f}s{t_naif*40/t_iter:>9.1f}x")print(f"{'2. pandas_udf par lot':<34}{t_lot:>9.1f}s{t_lot/t_iter:>9.1f}x")print(f"{'3. pandas_udf a iterateur':<34}{t_iter:>9.1f}s{1.0:>9.1f}x")

### Q3 *(5 pts)* —- **a.** Reportez vos trois mesures. Le classement correspond-il à votre prédiction ?- **b.** Combien de processus Python Spark a-t-il lancés ? Comment le vérifier ?- **c.** Expliquez précisément **pourquoi** la forme 3 ne charge le modèle qu'une fois par  processus. Qu'est-ce que la signature à itérateur permet, que l'autre ne permet pas ?- **d.** Faites varier `spark.sql.execution.arrow.maxRecordsPerBatch` de 256 à 32.  Quelle forme est la plus affectée, et pourquoi ?- **e.** Formulez la règle pratique en une phrase.

In [ ]:
# 3.5 — Pour la question d : réduire la taille des lotsspark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "32")spark.sparkContext.setJobDescription("G - lots de 32, sans iterateur")t_lot32, _ = chrono("forme 2 avec des lots de 32",                    lambda: images_pq.withColumn("c", classer_lot("image")).count())spark.sparkContext.setJobDescription("H - lots de 32, a iterateur")t_iter32, _ = chrono("forme 3 avec des lots de 32",                     lambda: images_pq.withColumn("c", classer_iter("image")).count())print(f"\nforme 2 : {t_lot:.1f}s -> {t_lot32:.1f}s  ({t_lot32/t_lot:.2f}x)")print(f"forme 3 : {t_iter:.1f}s -> {t_iter32:.1f}s  ({t_iter32/t_iter:.2f}x)")spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "256")

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — Classification et écriture en Iceberg  *(3 points)*

In [ ]:
# 4.1 — Le vrai modèle ONNX, si disponibleif MODE_REEL:    from PIL import Image    class ModeleONNX:        """MobileNetV2 : entree 1x3x224x224 float32, normalisation ImageNet."""        MOYENNE = np.array([0.485, 0.456, 0.406], dtype=np.float32)        ECART   = np.array([0.229, 0.224, 0.225], dtype=np.float32)        def __init__(self, chemin=CLASSIF):            self.session = ort.InferenceSession(                chemin, providers=["CPUExecutionProvider"])            self.entree = self.session.get_inputs()[0].name            etiq = f"{MODELES}/imagenet_classes.txt"            self.classes = ([l.strip() for l in open(etiq, encoding="utf-8")]                            if os.path.exists(etiq) else                            [f"classe_{i}" for i in range(1000)])        def _preparer(self, octets):            img = Image.open(io.BytesIO(octets)).convert("RGB").resize((224, 224))            x = np.asarray(img, dtype=np.float32) / 255.0        # 0-1            x = (x - self.MOYENNE) / self.ECART                  # normalisation ImageNet            return x.transpose(2, 0, 1)                          # HWC -> CHW        def predire(self, images_octets):            lot = np.stack([self._preparer(b) for b in images_octets])            sorties = self.session.run(None, {self.entree: lot})[0]            return [self.classes[int(i)] for i in sorties.argmax(axis=1)]        def predire_avec_score(self, images_octets):            lot = np.stack([self._preparer(b) for b in images_octets])            s = self.session.run(None, {self.entree: lot})[0]            e = np.exp(s - s.max(axis=1, keepdims=True))            p = e / e.sum(axis=1, keepdims=True)            return [(self.classes[int(i)], float(p[k, i]))                    for k, i in enumerate(p.argmax(axis=1))]    print("ModeleONNX defini")else:    print("mode substitution — on garde ModeleSubstitution")

In [ ]:
# 4.2 — Classifier avec score, en pandas_udf à itérateurSCHEMA_CLASSIF = StructType([    StructField("classe", StringType()),    StructField("score",  FloatType()),])@pandas_udf(SCHEMA_CLASSIF)def classifier(lots: Iterator[pd.Series]) -> Iterator[pd.DataFrame]:    m = charger_modele()    for lot in lots:        if MODE_REEL:            paires = m.predire_avec_score(list(lot))        else:            paires = [(c, 0.5) for c in m.predire(list(lot))]        yield pd.DataFrame({"classe": [p[0] for p in paires],                            "score":  [float(p[1]) for p in paires]})spark.sparkContext.setJobDescription("I - classification")resultats = (images_pq             .withColumn("prediction", classifier("image"))             .select("nom", "length",                     F.col("prediction.classe").alias("classe"),                     F.col("prediction.score").alias("score"))             .cache())t_classif, n = chrono("classification de 4 000 images", lambda: resultats.count())resultats.show(10, truncate=False)print(f"debit : {n/t_classif:.0f} images/s sur 4 coeurs, "      f"soit {n/t_classif/4:.0f} images/s/coeur")

In [ ]:
# 4.3 — Écrire dans une table Icebergspark.conf.set("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")spark.conf.set("spark.sql.catalog.lakehouse.type", "hadoop")spark.conf.set("spark.sql.catalog.lakehouse.warehouse", f"s3a://lakehouse/{UTILISATEUR}-tp11")spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.medias")spark.sql("DROP TABLE IF EXISTS lakehouse.medias.classifications")(resultats.select("nom", "classe", "score")          .writeTo("lakehouse.medias.classifications")          .using("iceberg").create())spark.sql("""SELECT classe, count(*) AS n, round(avg(score), 3) AS score_moyenFROM lakehouse.medias.classificationsGROUP BY classe ORDER BY n DESC""").show(15, truncate=False)

### Q4 *(3 pts)* —- **a.** Quel débit avez-vous mesuré, en images par seconde et par cœur ?- **b.** Comparez à l'ordre de grandeur annoncé en cours (20 à 50 images/s/cœur pour  MobileNetV2). Votre mesure est-elle cohérente ? Si vous êtes en mode substitution,  que mesurez-vous exactement ?- **c.** Les résultats sont maintenant dans une table Iceberg. Écrivez la requête SQL qui  retrouve les 10 images dont la prédiction est la moins sûre. À quoi cela peut-il servir ?

**Votre réponse :***(rédigez ici)*

---# Exercice 5 — Détection d'objets  *(3 points)***Objectif.** Passer d'une étiquette par image à *n* objets localisés — et voir ce que celachange pour le schéma et pour le coût.

In [ ]:
# 5.1 — Le schéma de sortie d'une détectionSCHEMA_DETECTION = ArrayType(StructType([    StructField("classe", StringType()),    StructField("score",  FloatType()),    StructField("x_min",  FloatType()),    StructField("y_min",  FloatType()),    StructField("x_max",  FloatType()),    StructField("y_max",  FloatType()),]))print("schema imbrique : array<struct<classe,score,x_min,y_min,x_max,y_max>>")

In [ ]:
# 5.2 — Détecteur (réel si disponible, sinon substitution)SEUIL_DETECTION = 0.35class DetecteurSubstitution:    """Retourne 1 a 3 boites deterministes, pour valider la MECANIQUE."""    def __init__(self, chemin=None):        time.sleep(COUT_CHARGEMENT_S * 1.5)      # un detecteur charge plus lentement        self.classes = ["cercle", "carre", "triangle", "etoile", "anneau"]    def detecter(self, images_octets):        time.sleep(len(images_octets) * COUT_IMAGE_MS * 5 / 1000.0)   # 5x plus lourd        sorties = []        for b in images_octets:            h = len(b) if b else 0            n = 1 + h % 3            sorties.append([{                "classe": self.classes[(h + i) % 5],                "score": round(0.5 + ((h + i) % 40) / 100.0, 3),                "x_min": round(((h + i) % 50) / 100.0, 3),                "y_min": round(((h + 2*i) % 50) / 100.0, 3),                "x_max": round(0.5 + ((h + i) % 45) / 100.0, 3),                "y_max": round(0.5 + ((h + 3*i) % 45) / 100.0, 3),            } for i in range(n)])        return sortiesdef charger_detecteur():    if MODE_REEL and os.path.exists(DETECT):        return DetecteurONNX(DETECT)    return DetecteurSubstitution()if MODE_REEL and os.path.exists(DETECT):    from PIL import Image    class DetecteurONNX:        """SSD-MobileNetV1 : entree 1xHxWx3 uint8, sorties boites/classes/scores/nb."""        def __init__(self, chemin=DETECT):            self.session = ort.InferenceSession(                chemin, providers=["CPUExecutionProvider"])            self.entree = self.session.get_inputs()[0].name        def detecter(self, images_octets):            sorties = []            for b in images_octets:                img = Image.open(io.BytesIO(b)).convert("RGB")                x = np.asarray(img, dtype=np.uint8)[np.newaxis, ...]  # NHWC uint8                boites, classes, scores, nb = self.session.run(None, {self.entree: x})                objets = []                for i in range(int(nb[0])):                    s = float(scores[0][i])                    if s < SEUIL_DETECTION:                        continue                    ymin, xmin, ymax, xmax = [float(v) for v in boites[0][i]]                    objets.append({"classe": f"coco_{int(classes[0][i])}", "score": s,                                   "x_min": xmin, "y_min": ymin,                                   "x_max": xmax, "y_max": ymax})                sorties.append(objets)            return sorties    print("DetecteurONNX defini")else:    print("detection en mode substitution")

In [ ]:
# 5.3 — Appliquer la détection, toujours en pandas_udf à itérateur@pandas_udf(SCHEMA_DETECTION)def detecter(lots: Iterator[pd.Series]) -> Iterator[pd.Series]:    d = charger_detecteur()    for lot in lots:        yield pd.Series(d.detecter(list(lot)))# On mesure sur un echantillon : la detection est bien plus lourdeechantillon2 = images_pq.limit(400).cache(); echantillon2.count()spark.sparkContext.setJobDescription("J - detection")t_detect, _ = chrono("detection sur 400 images",    lambda: echantillon2.withColumn("objets", detecter("image")).count())print(f"debit detection : {400/t_detect:.1f} images/s "      f"({400/t_detect/4:.1f} par coeur)")print(f"rapport avec la classification : "      f"{(t_detect/400)/(t_classif/4000):.1f}x plus lent par image")

In [ ]:
# 5.4 — Interroger les objets détectés, en SQLdetections = echantillon2.withColumn("objets", detecter("image")).cache()detections.select("nom", "objets").show(3, truncate=False)detections.createOrReplaceTempView("detections")print("\n--- nombre d'objets par image ---")spark.sql("""SELECT size(objets) AS nb_objets, count(*) AS nFROM detections GROUP BY size(objets) ORDER BY nb_objets""").show()print("--- images contenant un objet de score > 0,8 ---")spark.sql("""SELECT nom, size(objets) AS nbFROM detectionsWHERE exists(objets, o -> o.score > 0.8)LIMIT 5""").show(truncate=False)

### Q5 *(3 pts)* —- **a.** Combien de fois la détection est-elle plus lente que la classification, par image ?  Est-ce cohérent avec l'ordre de grandeur du cours (3 à 10) ?- **b.** Le schéma de sortie est `array<struct<...>>`. Quel mécanisme du module 2 permet à  Parquet de stocker cela nativement, sans aplatir ?- **c.** La requête 5.4 utilise `exists(objets, o -> o.score > 0.8)`. En quoi le fait de  pouvoir écrire cela en SQL change-t-il la nature du jeu de données ?

**Votre réponse :***(rédigez ici)*

---# Exercice 6 — Dimensionner, et le décalage de distribution  *(2 points)*

In [ ]:
# 6.1 — Dimensionner à partir de VOTRE mesuredebit_coeur = 4000 / t_classif / 4print(f"debit mesure : {debit_coeur:.1f} images/s/coeur")for n_images, n_executors, n_coeurs in [(5_000_000, 20, 5), (50_000_000, 60, 5)]:    coeurs = n_executors * n_coeurs    debit = coeurs * debit_coeur    duree = n_images / debit    debit_octets = debit * 200 * 1024        # images reelles de 200 Kio    print(f"\n{n_images:,} images sur {coeurs} coeurs")    print(f"  debit    : {debit:,.0f} images/s")    print(f"  duree    : {duree:,.0f} s  =  {duree/3600:.1f} h")    print(f"  stockage : {debit_octets/1024**2:,.0f} Mio/s a soutenir")

In [ ]:
# 6.2 — Que prédit le modèle, au juste ?verite = (spark.read.json(f"file:///home/tinku/work/images_verite.jsonl")          .select(F.col("fichier").alias("nom"),                  F.col("classe_dominante").alias("verite")))compare = resultats.join(verite, "nom")print("--- prediction du modele contre verite terrain ---")compare.groupBy("verite", "classe").count().orderBy(F.desc("count")).show(15, False)exact = compare.filter(F.col("classe") == F.col("verite")).count()print(f"\nexactitude : {exact}/{compare.count()} = "      f"{100*exact/max(compare.count(),1):.1f} %")print(f"score de confiance moyen : "      f"{compare.agg(F.avg('score')).collect()[0][0]:.3f}")

### Q6 *(2 pts)* —- **a.** À partir de votre débit mesuré : combien de temps pour 5 millions d'images sur  100 cœurs ? Le stockage suit-il ?- **b.** *(mode réel)* Quelle exactitude obtenez-vous contre la vérité terrain ? Quel est le  score de confiance moyen ? Ces deux chiffres vous paraissent-ils compatibles ?- **c.** Expliquez ce que vous observez. Comment un exploitant s'en apercevrait-il en  production, si personne ne disposait de la vérité terrain ?

**Votre réponse :***(rédigez ici)*

---# Synthèse| Mesure | Votre chiffre | Ce que vous en concluez ||---|---|---|| Tâches : fichiers contre Parquet | | || Forme 1 (UDF simple), extrapolée | | || Forme 2 (pandas_udf par lot) | | || Forme 3 (pandas_udf à itérateur) | | || Effet des lots de 32 sur les formes 2 et 3 | | || Débit mesuré, images/s/cœur | | || Exactitude contre score de confiance | | |**Question de conclusion.** Vous devez classifier 40 millions de photographies stockées enfichiers individuels sur S3. Rédigez en cinq lignes votre plan, dans l'ordre, en justifiantchaque étape par une mesure de ce TP.

**Votre réponse :***(rédigez ici)*

In [ ]:
# Nettoyagefor d in [echantillon, echantillon2, resultats, detections]:    try: d.unpersist()    except Exception: passspark.stop()print("Session fermee.")

---## Avant de rendre- [ ] Les **deux prédictions** (2.1 et 3.1) sont écrites **avant** exécution.- [ ] Les questions **Q1 à Q6** sont rédigées et justifiées.- [ ] Le mode utilisé — réel ou substitution — est indiqué en tête du notebook.- [ ] Le tableau de synthèse est complété.- [ ] Le plan de la question de conclusion est rédigé.- [ ] Notebook exporté en HTML et déposé.**Bon TP.**